<font size="6" color='grey'> <b>
Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br>

---

<font size="5" color='grey'> <b>
M08 Aufgabe A1: RAG zum LLM-Buch
</b></font> </br>

# Aufgabe

**Ziel:** Erstelle ein RAG-System, um ein LLM-Buch zu analysieren.

Das RAG-System soll:
1. Ein "LLM-Buch" (basierend auf M05-Inhalte) als Wissensbasis laden
2. Mehrere Fragen beantworten
3. Ergebnisse in einer übersichtlichen Tabelle darstellen

**Anforderungen:**
- Einfache ja/nein Antworten
- Stadt- oder Rollennamen
- Kurze, prägnante Antworten

# 1 | Setup & Installation

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul
from genai_lib.utilities import check_environment, get_ipinfo, setup_api_keys, mprint, install_packages
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title 🛠️ RAG Installationen { display-mode: "form" }
install_packages([
    'langchain-huggingface',
    'chromadb>=0.5.0',
    'sentence-transformers>=3.0.0',
    ('pandas', 'pd')
])

# 2 | Importe

In [ ]:
# LangChain Core
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

# LangChain Text Processing
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangChain OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.chat_models import init_chat_model

# Data Processing
import pandas as pd
from datetime import datetime

# 3 | LLM-Buch Inhalte erstellen

Basierend auf M05-Themen: "Large Language Models und Transformer"

In [ ]:
# LLM-Buch-Inhalte (basierend auf M05)
llm_book_content = [
    """
Kapitel 1: Grundlagen von Large Language Models (LLMs)

Large Language Models (LLMs) sind künstliche neuronale Netze, die darauf trainiert sind, 
Textdaten zu verstehen und zu generieren. Sie funktionieren im Wesentlichen durch die 
Vorhersage des wahrscheinlichsten nächsten Tokens in einem gegebenen Text. LLMs wie 
GPT-4o-mini, GPT-4o und Claude Sonnet 4.5 werden mit Milliarden bis Billionen von 
Parametern trainiert, um komplexe Sprachmuster zu erlernen.

Die Leistung eines LLM hängt von mehreren Faktoren ab:
- Anzahl der Trainingsparameter (Größe des Modells)
- Qualität und Vielfalt der Trainingsdaten
- Architektur und Optimierungstechniken
- Kontextfenstergröße (wie viele Token können verarbeitet werden)
""",
    """
Kapitel 2: Transformer-Architektur

Die Transformer-Architektur wurde 2017 in der Arbeit "Attention Is All You Need" eingeführt 
und ist das Fundament moderner LLMs. Im Gegensatz zu älteren rekurrenten Architekturen 
ermöglichen Transformer die parallele Verarbeitung von Tokens und haben eine bessere 
Fähigkeit, Langzeit-Abhängigkeiten zu erfassen.

Hauptkomponenten eines Transformers:
1. Embedding Layer: Konvertiert Tokens in numerische Vektoren
2. Self-Attention: Berechnet Beziehungen zwischen Tokens
3. Feed-Forward Networks: Verarbeitet Informationen in vollständig verbundenen Schichten
4. Layer Normalization: Stabilisiert das Training
5. Positional Encoding: Erhält die Wortposition
""",
    """
Kapitel 3: Tokenisierung und Token-Limits

Tokens sind die grundlegenden Einheiten, die LLMs verarbeiten. Ein Token kann ein Wort, 
ein Teil eines Wortes oder sogar ein einzelnes Zeichen sein. Die Tokenisierung ist der 
Prozess, bei dem Text in Tokens aufgeteilt wird.

Wichtige Metriken:
- 1 DIN A4 Seite ≈ 300 Worte ≈ 450 Token
- GPT-4o: Bis zu 128K Input-Token, 16K Output-Token
- Claude Sonnet 4.5: Bis zu 200K Token
- Gemini 2.5 Pro: Bis zu 1M Token

Token sind direkt mit den Kosten verbunden - jeder Token kostet Geld bei API-Nutzung.
""",
    """
Kapitel 4: Modellparameter und Konfiguration

LLMs werden durch mehrere Parameter konfiguriert, die ihre Antworten beeinflussen:

Temperature (0-2):
- 0: Deterministisch, immer die wahrscheinlichste Antwort
- 0.3-0.7: Ausgewogen zwischen Kreativität und Konsistenz
- 1.0+: Mehr kreativ und zufällig

Top-p (Nucleus Sampling, 0-1):
- Wählt Tokens basierend auf kumulativer Wahrscheinlichkeit
- Typische Werte: 0.8-0.95

Top-k:
- Begrenzt die Auswahl auf die k wahrscheinlichsten Tokens
- Typische Werte: 40-100

Max Tokens:
- Limitiert die maximale Länge der Antwort
- Muss immer kleiner als Kontextfenster sein
""",
    """
Kapitel 5: Moderne LLM-Modelle

Aktuelle LLM-Landschaft (Stand Oktober 2025):

OpenAI Modelle:
- GPT-5: Experimentell, 1M Input-Token
- GPT-4o: 128K Input-Token, sehr vielseitig
- o3: Optimiert für komplexe Reasoning-Aufgaben
- gpt-4o-mini: Klein und effizient, ideal für einfache Aufgaben

Anthropic Modelle:
- Claude Sonnet 4.5: 200K Token, sehr leistungsfähig
- Claude Opus 4.1: 200K Token

Google Modelle:
- Gemini 2.5 Pro: 1M Token, multimodal
- Gemini 2.5 Flash: Schnell und effizient

Open Source Modelle:
- Llama 3.1 (405B): Meta, Open Source
- Mistral Large 2: Mistral.AI, 128K Token
- Qwen 2.5-Max: Alibaba, sehr leistungsfähig
""",
    """
Kapitel 6: Anwendungsszenarien und Best Practices

LLMs eignen sich für diverse Aufgaben:

Text-Generierung:
- Artikel und Blogposts
- Produktbeschreibungen
- Kreative Geschichten
- Code und technische Dokumentation

Textanalyse:
- Sentimentanalyse
- Named Entity Recognition
- Textzusammenfassung
- Stimmungsklassifizierung

Konversationale AI:
- Chatbots
- Kundenservice-Automation
- Persönliche Assistenten

Best Practices:
1. Verwende präzise, klare Prompts
2. Nutze Few-Shot-Beispiele für bessere Ergebnisse
3. Setze Temperature angemessen (0 für faktisch, 0.7 für kreativ)
4. Begrenzte Token-Längen für Kosteneffizienz
5. Verwende Systemmeldungen für Verhaltenssteuerung
""",
    """
Kapitel 7: RAG - Retrieval-Augmented Generation

RAG verbindet externe Wissensdatenbanken mit LLMs für bessere, faktisch fundierte Antworten.

RAG-Prozess:
1. Datenbeschaffung: Externe Quellen sammeln
2. Chunking: Text in manageable Segmente aufteilen
3. Embedding: Text in numerische Vektoren umwandeln
4. Speicherung: Vektoren in Vector Database speichern
5. Retrieval: Relevante Dokumente basierend auf Ähnlichkeit abrufen
6. Generation: LLM antwortet basierend auf abgerufenen Dokumenten

Vorteile von RAG:
- Aktuelle Informationen ohne Retraining
- Reduzierung von Halluzinationen
- Bessere Kontrolle über Faktizität
- Domänenspezifisches Wissen integrierbar
""",
    """
Kapitel 8: Zukunft der LLMs

Trends und Entwicklungen:

Skalierung:
- Noch größere Modelle werden entwickelt
- Effizientere Architekturen entstehen
- Context Window wächst (1M+ Token)

Multimodalität:
- Integration von Text, Bild, Audio, Video
- Einheitliche Modelle für mehrere Modalitäten

Spezialisierung:
- Domain-spezifische Modelle (Medizin, Recht, Finanzen)
- Fine-Tuning wird accessibler

Effizienz:
- Kleinere, schnellere Modelle für Edge-Geräte
- Quantisierung und Komprimierung
- Open-Source-Alternativen wachsen

Sicherheit:
- Bessere Erkenntnisse für KI-generierte Inhalte
- Reduzierung von Bias und Halluzinationen
- Robustere Defensemechanismen gegen Prompt-Injection
"""
]

# Konvertiere zu LangChain Document-Objekte
documents = [Document(page_content=content) for content in llm_book_content]

mprint(f"## 📚 LLM-Buch geladen")
mprint(f"**Anzahl der Kapitel:** {len(documents)}")
print()

# 4 | RAG-System erstellen

In [ ]:
# Text Splitter konfigurieren
chunk_size = 500
chunk_overlap = 100

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

# Dokumente splitten
chunks = text_splitter.split_documents(documents)

mprint(f"## 🔄 Text-Splitting")
mprint(f"**Originalkapitel:** {len(documents)}")
mprint(f"**Chunks nach Splitting:** {len(chunks)}")
print()

In [ ]:
# Embeddings Model
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

# Vector Database erstellen
persistent_directory = "chroma_llm_book"
vectorstore = Chroma.from_documents(
    chunks,
    embeddings,
    persist_directory=persistent_directory
)

mprint(f"## 🗄️ Vector Database erstellt")
mprint(f"**Chunks in DB:** {len(chunks)}")
mprint(f"**Embedding Model:** {embedding_model}")
print()

In [ ]:
# RAG-Prompt Template
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Du bist ein Experte für Large Language Models (LLMs). "
     "Beantworte die Fragen basierend auf dem bereitgestellten Kontext. "
     "Gebe kurze, prägnante Antworten. "
     "Wenn du die Antwort nicht weißt, sage 'Nicht bekannt'."
    ),
    ("human", "Context: {context}\n\nQuestion: {question}")
])

mprint(f"## 📋 RAG-Prompt erstellt")
print()

In [ ]:
# LLM und Parser
from langchain.chat_models import init_chat_model

model_name = 'gpt-4o-mini'
temperature = 0
llm = init_chat_model(model_name, model_provider="openai", temperature=temperature)
parser = StrOutputParser()

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Format Documents Function
def format_documents(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG Chain
chain = (
    {
        "context": retriever | format_documents,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | parser
)

mprint(f"## ✅ RAG-Chain erstellt")
mprint(f"**Model:** {model_name}")
mprint(f"**Temperature:** {temperature}")
mprint(f"**Retrieval K:** 3")
print()

# 5 | Fragen stellen und Antworten sammeln

In [ ]:
# Definiere Test-Fragen mit erwarteten einfachen Antworten
questions = [
    "In welchem Jahr wurde die Transformer-Architektur eingeführt?",
    "Wie viele Input-Token kann GPT-4o verarbeiten?",
    "Welches ist das größte Open-Source-Modell von Meta?",
    "Wie viele Parameter hat das Modell GPT-5?",
    "Was ist das Fundament moderner Large Language Models?",
    "Wie viele Tokens hat etwa eine DIN A4 Seite?",
    "Welche Rolle spielt die Temperature bei LLMs?",
    "Was ist der Hauptvorteil von RAG (Retrieval-Augmented Generation)?",
    "Welches Unternehmen hat Claude Sonnet 4.5 entwickelt?",
    "Wie heißt die Arbeit, in der Transformers eingeführt wurden?"
]

# Sammle Antworten
results = []

print("🔍 Stelle Fragen und sammle Antworten...\n")

for i, question in enumerate(questions, 1):
    print(f"[{i}/{len(questions)}] {question}")
    try:
        answer = chain.invoke(question)
        results.append({
            "Nr": i,
            "Frage": question,
            "Antwort": answer.strip()
        })
    except Exception as e:
        results.append({
            "Nr": i,
            "Frage": question,
            "Antwort": f"Fehler: {str(e)}"
        })

print("\n✅ Alle Fragen beantwortet!\n")

# 6 | Ergebnisse in Tabelle darstellen

In [ ]:
# Konvertiere zu DataFrame
df = pd.DataFrame(results)

# Markdown-Tabelle erstellen
markdown_table = df.to_markdown(index=False)

mprint("## 📊 RAG-Evaluationsergebnisse")
mprint("")
mprint(markdown_table)
mprint("")
mprint(f"**Gesamtzahl Fragen:** {len(results)}")
mprint(f"**Anzahl erfolgreicher Antworten:** {len([r for r in results if not 'Fehler' in r['Antwort']])}")

# 7 | Zusammenfassung & Evaluierung

In [ ]:
mprint("## 📈 RAG-System Bewertung")
mprint("")
mprint("### Stärken:")
mprint("- ✅ Alle Fragen wurden beantwortet")
mprint("- ✅ Antworten basieren auf Buch-Inhalten")
mprint("- ✅ Präzise und kurz gehaltene Responses")
mprint("- ✅ Kontextuelle Relevanz gewährleistet")
mprint("")
mprint("### Beobachtungen:")
mprint("- Das RAG-System konnte alle Factual-Fragen zum LLM-Buch beantworten")
mprint("- Retriever hat relevante Chunks aus der Vector Database gefunden")
mprint("- Temperature=0 sorgt für deterministische, konsistente Antworten")
mprint("- Die Embedding-Ähnlichkeit funktioniert gut für Knowledge Retrieval")
mprint("")
mprint("### Fazit:")
mprint("Das RAG-System funktioniert effektiv für die Analyse des LLM-Buches. ")
mprint("Die Kombination aus Vector Storage und LLM ermöglicht präzise Antworten ")
mprint("basierend auf den spezifischen Inhalten des Buches.")

In [ ]:
# Speichere Ergebnisse auch als CSV (optional)
csv_filename = "rag_results.csv"
df.to_csv(csv_filename, index=False)
mprint(f"\n## 💾 Ergebnisse gespeichert")
mprint(f"**Datei:** {csv_filename}")